## SQL - preapre_data.py

PostgreSQL

Open-Meteo → validated UTC weather data → persistent canonical PostgreSQL table → exercise-specific practice table

### Structure

1. Configuration

2. Disposable Development Database
   reset_dev_database()
   verify database + UTC

3. Time Contract
   interval_contract()
   ordinary / spring DST / fall DST proof

4. Source
   fetch_weather_interval()
   one 2025 source proof

5. Canonical Storage
   EXPECTED_CANONICAL_COLUMNS
   ensure_canonical_storage()
   validate_canonical_storage()
   one compact transaction rollback proof

6. Canonical Load
   create_staging_table()
   copy_to_staging()
   validate_staging()
   load_canonical()
   one corruption/rollback proof
   valid 2025 load
   same load again → 0

7. Incremental Coverage
   short markdown about discarded year-based design
   get_canonical_times()
   coverage_diff()
   timestamps_to_intervals()
   find_missing_intervals()
   validate_coverage()
   split_request_interval()
   fetch_missing_intervals()

   inspect missing interval
   fetch it
   load it
   validate coverage

8. Practice Materialization
   materialize_practice_table()
   validate_practice_table()
   materialize
   query back

9. Stable Orchestration
   ensure_database()
   main()

10. End-to-End
    reset_dev_database()
    main()       # first run
    main()       # rerun

11. Current prepare_data.py
    entire standalone copy/paste block

In [1]:
import time

import pandas as pd
import psycopg
import requests
from psycopg import sql


DATABASE_NAME = "analytical_workflow_fluency_dev"

CANONICAL_SCHEMA = "canonical"
CANONICAL_TABLE = "weather_hourly"

PRACTICE_SCHEMA = "practice_008"
PRACTICE_TABLE = "weather_hourly"

LOCAL_TIMEZONE = "America/Edmonton"

PRACTICE_START_LOCAL = pd.Timestamp(
    "2024-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

PRACTICE_END_LOCAL = pd.Timestamp(
    "2026-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

LATITUDE = 51.0447
LONGITUDE = -114.0719
MODEL = "era5"

# Empirical pacing from prior development; not an API guarantee.
REQUEST_DELAY_SECONDS = 4

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "pressure_msl",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
    "vapour_pressure_deficit",
    "soil_temperature_0_to_7cm",
]

COLUMNS = [
    "time",
    *HOURLY_VARIABLES,
]

In [2]:
# This notebook uses a disposable database so Restart Kernel → Run All starts from known state. Reset function is development-only.

def reset_dev_database():
    if not DATABASE_NAME.endswith("_dev"):
        raise RuntimeError(
            "Refusing to reset a non-dev database."
        )

    with psycopg.connect(
        dbname="postgres",
        autocommit=True,
    ) as conn:
        conn.execute(
            '''
            SELECT pg_terminate_backend(pid)
            FROM pg_stat_activity
            WHERE datname = %s
              AND pid <> pg_backend_pid();
            ''',
            (DATABASE_NAME,),
        )

        conn.execute(
            sql.SQL(
                "DROP DATABASE IF EXISTS {}"
            ).format(
                sql.Identifier(DATABASE_NAME)
            )
        )

        conn.execute(
            sql.SQL(
                "CREATE DATABASE {}"
            ).format(
                sql.Identifier(DATABASE_NAME)
            )
        )

        conn.execute(
            sql.SQL('''
                ALTER DATABASE {}
                SET timezone TO {}
            ''').format(
                sql.Identifier(DATABASE_NAME),
                sql.Literal("UTC"),
            )
        )


reset_dev_database()

In [3]:
with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    database_state = conn.execute(
        '''
        SELECT
            current_database(),
            current_setting('TimeZone');
        '''
    ).fetchone()

database_state

('analytical_workflow_fluency_dev', 'UTC')

In [4]:
# Canonical timestamps are UTC

def interval_contract(
    start_local,
    end_local,
):
    start_local = pd.Timestamp(start_local)
    end_local = pd.Timestamp(end_local)

    if (
        start_local.tz is None
        or end_local.tz is None
    ):
        raise ValueError(
            "Interval boundaries must be timezone-aware."
        )

    if start_local >= end_local:
        raise ValueError(
            "Interval start must precede end."
        )

    start_utc = start_local.tz_convert("UTC")
    end_utc = end_local.tz_convert("UTC")

    expected_times = pd.date_range(
        start=start_utc,
        end=end_utc,
        freq="h",
        inclusive="left",
    )

    return (
        start_utc,
        end_utc,
        expected_times,
    )


time_tests = [
    (
        "ordinary",
        pd.Timestamp(
            "2025-01-15 00:00",
            tz=LOCAL_TIMEZONE,
        ),
        pd.Timestamp(
            "2025-01-17 00:00",
            tz=LOCAL_TIMEZONE,
        ),
    ),
    (
        "spring DST",
        pd.Timestamp(
            "2025-03-09 00:00",
            tz=LOCAL_TIMEZONE,
        ),
        pd.Timestamp(
            "2025-03-10 00:00",
            tz=LOCAL_TIMEZONE,
        ),
    ),
    (
        "fall DST",
        pd.Timestamp(
            "2025-11-02 00:00",
            tz=LOCAL_TIMEZONE,
        ),
        pd.Timestamp(
            "2025-11-03 00:00",
            tz=LOCAL_TIMEZONE,
        ),
    ),
]

for name, start_local, end_local in time_tests:
    start_utc, end_utc, expected = interval_contract(
        start_local,
        end_local,
    )

    print(
        name,
        start_utc,
        end_utc,
        len(expected),
    )

ordinary 2025-01-15 07:00:00+00:00 2025-01-17 07:00:00+00:00 48
spring DST 2025-03-09 07:00:00+00:00 2025-03-10 06:00:00+00:00 23
fall DST 2025-11-02 06:00:00+00:00 2025-11-03 07:00:00+00:00 25


In [5]:
# Fetch in UTC, parse Unix timestamps as UTC-aware datetimes, filter to exact requeested initernvval, and reject structural or timestamp-population failures

def fetch_weather_interval(
    start_local,
    end_local,
):
    (
        start_utc,
        end_utc,
        expected_times,
    ) = interval_contract(
        start_local,
        end_local,
    )

    api_start_date = start_utc.date().isoformat()

    api_end_date = (
        end_utc
        - pd.Timedelta(hours=1)
    ).date().isoformat()

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": api_start_date,
        "end_date": api_end_date,
        "hourly": ",".join(
            HOURLY_VARIABLES
        ),
        "models": MODEL,
        "timezone": "GMT",
        "timeformat": "unixtime",
    }

    response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    payload = response.json()

    if (
        payload.get("timezone") != "GMT"
        or payload.get(
            "utc_offset_seconds"
        ) != 0
    ):
        raise ValueError(
            "Unexpected source timezone."
        )

    hourly = payload.get("hourly")

    if not isinstance(hourly, dict):
        raise ValueError(
            "Missing hourly data."
        )

    expected_keys = {
        "time",
        *HOURLY_VARIABLES,
    }

    if set(hourly) != expected_keys:
        raise ValueError(
            "Unexpected hourly fields."
        )

    lengths = {
        len(values)
        for values in hourly.values()
    }

    if len(lengths) != 1:
        raise ValueError(
            "Hourly arrays have unequal lengths."
        )

    frame = pd.DataFrame(hourly)

    frame["time"] = pd.to_datetime(
        frame["time"],
        unit="s",
        utc=True,
    )

    frame = (
        frame
        .loc[
            frame["time"].ge(start_utc)
            & frame["time"].lt(end_utc),
            COLUMNS,
        ]
        .reset_index(drop=True)
    )

    actual_times = pd.DatetimeIndex(
        frame["time"]
    )

    missing_times = expected_times.difference(
        actual_times
    )
    unexpected_times = actual_times.difference(
        expected_times
    )

    if (
        frame["time"].isna().any()
        or not frame["time"].is_unique
        or not frame[
            "time"
        ].is_monotonic_increasing
        or len(missing_times) > 0
        or len(unexpected_times) > 0
    ):
        raise ValueError(
            "Source timestamp validation failed."
        )

    return frame

In [6]:
year_2025_start = pd.Timestamp(
    "2025-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

year_2025_end = pd.Timestamp(
    "2026-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

weather_2025 = fetch_weather_interval(
    year_2025_start,
    year_2025_end,
)

{
    "rows": len(weather_2025),
    "start": weather_2025["time"].min(),
    "end": weather_2025["time"].max(),
    "time_dtype": str(
        weather_2025["time"].dtype
    ),
    "measurement_nulls": int(
        weather_2025[
            HOURLY_VARIABLES
        ]
        .isna()
        .sum()
        .sum()
    ),
}

{'rows': 8760,
 'start': Timestamp('2025-01-01 07:00:00+0000', tz='UTC'),
 'end': Timestamp('2026-01-01 06:00:00+0000', tz='UTC'),
 'time_dtype': 'datetime64[s, UTC]',
 'measurement_nulls': 0}

In [7]:
# The canonical table is persistent and trusted. Explicit schema contract.

EXPECTED_CANONICAL_COLUMNS = [
    ("time", "timestamp with time zone", "NO"),
    ("temperature_2m", "double precision", "YES"),
    ("relative_humidity_2m", "double precision", "YES"),
    ("dew_point_2m", "double precision", "YES"),
    ("precipitation", "double precision", "YES"),
    ("rain", "double precision", "YES"),
    ("snowfall", "double precision", "YES"),
    ("weather_code", "integer", "YES"),
    ("cloud_cover", "double precision", "YES"),
    ("pressure_msl", "double precision", "YES"),
    ("surface_pressure", "double precision", "YES"),
    ("wind_speed_10m", "double precision", "YES"),
    ("wind_direction_10m", "double precision", "YES"),
    ("shortwave_radiation", "double precision", "YES"),
    ("vapour_pressure_deficit", "double precision", "YES"),
    ("soil_temperature_0_to_7cm", "double precision", "YES"),
]


def ensure_canonical_storage(conn):
    conn.execute(
        sql.SQL('''
            CREATE SCHEMA IF NOT EXISTS {}
        ''').format(
            sql.Identifier(
                CANONICAL_SCHEMA
            )
        )
    )

    conn.execute(
        sql.SQL('''
            CREATE TABLE IF NOT EXISTS {}.{} (
                time TIMESTAMPTZ PRIMARY KEY,
                temperature_2m DOUBLE PRECISION,
                relative_humidity_2m DOUBLE PRECISION,
                dew_point_2m DOUBLE PRECISION,
                precipitation DOUBLE PRECISION,
                rain DOUBLE PRECISION,
                snowfall DOUBLE PRECISION,
                weather_code INTEGER,
                cloud_cover DOUBLE PRECISION,
                pressure_msl DOUBLE PRECISION,
                surface_pressure DOUBLE PRECISION,
                wind_speed_10m DOUBLE PRECISION,
                wind_direction_10m DOUBLE PRECISION,
                shortwave_radiation DOUBLE PRECISION,
                vapour_pressure_deficit DOUBLE PRECISION,
                soil_temperature_0_to_7cm DOUBLE PRECISION
            )
        ''').format(
            sql.Identifier(
                CANONICAL_SCHEMA
            ),
            sql.Identifier(
                CANONICAL_TABLE
            ),
        )
    )


def validate_canonical_storage(conn):
    columns = conn.execute(
        '''
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = %s
          AND table_name = %s
        ORDER BY ordinal_position;
        ''',
        (
            CANONICAL_SCHEMA,
            CANONICAL_TABLE,
        ),
    ).fetchall()

    if (
        columns
        != EXPECTED_CANONICAL_COLUMNS
    ):
        raise ValueError(
            "Canonical column contract mismatch."
        )

    primary_key_columns = (
        conn.execute(
            '''
            SELECT
                kcu.column_name
            FROM information_schema.table_constraints AS tc
            JOIN information_schema.key_column_usage AS kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            WHERE tc.constraint_type = 'PRIMARY KEY'
              AND tc.table_schema = %s
              AND tc.table_name = %s
            ORDER BY kcu.ordinal_position;
            ''',
            (
                CANONICAL_SCHEMA,
                CANONICAL_TABLE,
            ),
        )
        .fetchall()
    )

    if (
        primary_key_columns
        != [("time",)]
    ):
        raise ValueError(
            "Canonical primary-key contract mismatch."
        )

In [8]:
# Context-manager transaction behavior used by the stable script.
with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    conn.execute(
        '''
        CREATE TABLE transaction_test (
            id INTEGER PRIMARY KEY
        );
        '''
    )

try:
    with psycopg.connect(
        dbname=DATABASE_NAME
    ) as conn:
        conn.execute(
            '''
            INSERT INTO transaction_test
            VALUES (1);
            '''
        )

        raise RuntimeError(
            "rollback test"
        )

except RuntimeError:
    pass

with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    transaction_test_count = (
        conn.execute(
            '''
            SELECT COUNT(*)
            FROM transaction_test;
            '''
        )
        .fetchone()[0]
    )

    conn.execute(
        '''
        DROP TABLE transaction_test;
        '''
    )

with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    ensure_canonical_storage(conn)
    validate_canonical_storage(conn)

transaction_test_count

0

In [9]:
# Canonical Load

STAGING_TABLE = "staging_weather"


def create_staging_table(conn):
    conn.execute(
        sql.SQL('''
            CREATE TEMP TABLE {}
            ON COMMIT DROP
            AS
            SELECT *
            FROM {}.{}
            WITH NO DATA
        ''').format(
            sql.Identifier(
                STAGING_TABLE
            ),
            sql.Identifier(
                CANONICAL_SCHEMA
            ),
            sql.Identifier(
                CANONICAL_TABLE
            ),
        )
    )


def copy_to_staging(
    conn,
    frame,
):
    column_sql = sql.SQL(", ").join(
        sql.Identifier(column)
        for column in COLUMNS
    )

    copy_statement = sql.SQL('''
        COPY {} ({})
        FROM STDIN
    ''').format(
        sql.Identifier(
            STAGING_TABLE
        ),
        column_sql,
    )

    with conn.cursor() as cur:
        with cur.copy(
            copy_statement
        ) as copy:
            for row in frame[
                COLUMNS
            ].itertuples(
                index=False,
                name=None,
            ):
                output_row = []

                for (
                    column,
                    value,
                ) in zip(
                    COLUMNS,
                    row,
                ):
                    if pd.isna(value):
                        output_row.append(
                            None
                        )

                    elif column == "time":
                        output_row.append(
                            value
                            .to_pydatetime()
                        )

                    elif (
                        column
                        == "weather_code"
                    ):
                        numeric_value = float(
                            value
                        )

                        if not (
                            numeric_value
                            .is_integer()
                        ):
                            raise ValueError(
                                "weather_code is not integral."
                            )

                        output_row.append(
                            int(
                                numeric_value
                            )
                        )

                    else:
                        output_row.append(
                            float(value)
                        )

                copy.write_row(
                    tuple(
                        output_row
                    )
                )


def validate_staging(
    conn,
    expected_rows,
):
    row_count = (
        conn.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM {}
            ''').format(
                sql.Identifier(
                    STAGING_TABLE
                )
            )
        )
        .fetchone()[0]
    )

    null_keys = (
        conn.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM {}
                WHERE time IS NULL
            ''').format(
                sql.Identifier(
                    STAGING_TABLE
                )
            )
        )
        .fetchone()[0]
    )

    duplicate_keys = (
        conn.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM (
                    SELECT time
                    FROM {}
                    GROUP BY time
                    HAVING COUNT(*) > 1
                ) AS duplicates
            ''').format(
                sql.Identifier(
                    STAGING_TABLE
                )
            )
        )
        .fetchone()[0]
    )

    if (
        row_count
        != expected_rows
        or null_keys != 0
        or duplicate_keys != 0
    ):
        raise ValueError(
            "Staging validation failed."
        )


def load_canonical(
    conn,
    frame,
):
    create_staging_table(conn)

    copy_to_staging(
        conn,
        frame,
    )

    validate_staging(
        conn,
        len(frame),
    )

    column_sql = sql.SQL(", ").join(
        sql.Identifier(column)
        for column in COLUMNS
    )

    result = conn.execute(
        sql.SQL('''
            INSERT INTO {}.{} ({})
            SELECT {}
            FROM {}
            ON CONFLICT (time) DO NOTHING
        ''').format(
            sql.Identifier(
                CANONICAL_SCHEMA
            ),
            sql.Identifier(
                CANONICAL_TABLE
            ),
            column_sql,
            column_sql,
            sql.Identifier(
                STAGING_TABLE
            ),
        )
    )

    return result.rowcount

In [10]:
canonical_before = 0

with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    canonical_before = (
        conn.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM {}.{}
            ''').format(
                sql.Identifier(
                    CANONICAL_SCHEMA
                ),
                sql.Identifier(
                    CANONICAL_TABLE
                ),
            )
        )
        .fetchone()[0]
    )

try:
    with psycopg.connect(
        dbname=DATABASE_NAME
    ) as conn:
        create_staging_table(conn)

        sample = (
            weather_2025
            .iloc[:48]
            .copy()
        )

        copy_to_staging(
            conn,
            sample,
        )

        deleted_time = sample[
            "time"
        ].iloc[12]

        conn.execute(
            sql.SQL('''
                DELETE FROM {}
                WHERE time = %s
            ''').format(
                sql.Identifier(
                    STAGING_TABLE
                )
            ),
            (
                deleted_time
                .to_pydatetime(),
            ),
        )

        conn.execute(
            sql.SQL('''
                INSERT INTO {}
                SELECT *
                FROM {}
                ORDER BY time
                LIMIT 1
            ''').format(
                sql.Identifier(
                    STAGING_TABLE
                ),
                sql.Identifier(
                    STAGING_TABLE
                ),
            )
        )

        validate_staging(
            conn,
            len(sample),
        )

except ValueError as error:
    print(
        type(error).__name__
    )

with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    canonical_after = (
        conn.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM {}.{}
            ''').format(
                sql.Identifier(
                    CANONICAL_SCHEMA
                ),
                sql.Identifier(
                    CANONICAL_TABLE
                ),
            )
        )
        .fetchone()[0]
    )

canonical_before, canonical_after

ValueError


(0, 0)

In [11]:
with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    first_insert = load_canonical(
        conn,
        weather_2025,
    )

with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    second_insert = load_canonical(
        conn,
        weather_2025,
    )

first_insert, second_insert

(8760, 0)

In [12]:
# Whole calendar years is too coarse for concurrent extentions. Instead we use exact required timestamp for interval, with years only for batching.

def get_canonical_times(
    conn,
    start_utc,
    end_utc,
):
    rows = conn.execute(
        sql.SQL('''
            SELECT time
            FROM {}.{}
            WHERE time >= %s
              AND time < %s
            ORDER BY time
        ''').format(
            sql.Identifier(
                CANONICAL_SCHEMA
            ),
            sql.Identifier(
                CANONICAL_TABLE
            ),
        ),
        (
            start_utc
            .to_pydatetime(),
            end_utc
            .to_pydatetime(),
        ),
    ).fetchall()

    if not rows:
        return pd.DatetimeIndex(
            [],
            tz="UTC",
        )

    return (
        pd.DatetimeIndex(
            [
                row[0]
                for row in rows
            ]
        )
        .tz_convert("UTC")
    )


def coverage_diff(
    conn,
    start_local,
    end_local,
):
    (
        start_utc,
        end_utc,
        expected_times,
    ) = interval_contract(
        start_local,
        end_local,
    )

    actual_times = (
        get_canonical_times(
            conn,
            start_utc,
            end_utc,
        )
    )

    missing_times = (
        expected_times
        .difference(actual_times)
        .sort_values()
    )

    unexpected_times = (
        actual_times
        .difference(expected_times)
        .sort_values()
    )

    return (
        missing_times,
        unexpected_times,
    )


def timestamps_to_intervals(
    timestamps,
):
    if len(timestamps) == 0:
        return []

    timestamps = (
        pd.DatetimeIndex(
            timestamps
        )
        .sort_values()
    )

    intervals = []
    interval_start = timestamps[0]
    previous = timestamps[0]

    for timestamp in timestamps[1:]:
        if (
            timestamp - previous
            != pd.Timedelta(hours=1)
        ):
            intervals.append(
                (
                    interval_start,
                    previous
                    + pd.Timedelta(
                        hours=1
                    ),
                )
            )

            interval_start = timestamp

        previous = timestamp

    intervals.append(
        (
            interval_start,
            previous
            + pd.Timedelta(
                hours=1
            ),
        )
    )

    return intervals


def find_missing_intervals(
    conn,
    start_local,
    end_local,
):
    (
        missing_times,
        unexpected_times,
    ) = coverage_diff(
        conn,
        start_local,
        end_local,
    )

    if len(
        unexpected_times
    ) > 0:
        raise ValueError(
            "Canonical contains unexpected timestamps."
        )

    return timestamps_to_intervals(
        missing_times
    )


def validate_coverage(
    conn,
    start_local,
    end_local,
):
    (
        missing_times,
        unexpected_times,
    ) = coverage_diff(
        conn,
        start_local,
        end_local,
    )

    if (
        len(missing_times) > 0
        or len(
            unexpected_times
        ) > 0
    ):
        raise ValueError(
            "Canonical coverage is invalid."
        )

In [13]:
def split_request_interval(
    start_utc,
    end_utc,
):
    start_local = (
        start_utc
        .tz_convert(
            LOCAL_TIMEZONE
        )
    )

    end_local = (
        end_utc
        .tz_convert(
            LOCAL_TIMEZONE
        )
    )

    batches = []
    current_start = start_local

    while current_start < end_local:
        next_year = pd.Timestamp(
            f"{current_start.year + 1}-01-01 00:00",
            tz=LOCAL_TIMEZONE,
        )

        batch_end = min(
            next_year,
            end_local,
        )

        batches.append(
            (
                current_start,
                batch_end,
            )
        )

        current_start = batch_end

    return batches


def fetch_missing_intervals(
    missing_intervals,
):
    request_intervals = []

    for (
        start_utc,
        end_utc,
    ) in missing_intervals:
        request_intervals.extend(
            split_request_interval(
                start_utc,
                end_utc,
            )
        )

    if not request_intervals:
        return None

    frames = []

    for index, (
        start_local,
        end_local,
    ) in enumerate(
        request_intervals
    ):
        print(
            "Fetching:",
            start_local,
            "to",
            end_local,
        )

        frames.append(
            fetch_weather_interval(
                start_local,
                end_local,
            )
        )

        if (
            index
            < len(
                request_intervals
            ) - 1
        ):
            time.sleep(
                REQUEST_DELAY_SECONDS
            )

    frame = pd.concat(
        frames,
        ignore_index=True,
    )

    if (
        frame["time"]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate timestamps across source batches."
        )

    return frame

In [14]:
with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    missing_intervals = (
        find_missing_intervals(
            conn,
            PRACTICE_START_LOCAL,
            PRACTICE_END_LOCAL,
        )
    )

missing_intervals

[(Timestamp('2024-01-01 07:00:00+0000', tz='UTC'),
  Timestamp('2025-01-01 07:00:00+0000', tz='UTC'))]

In [15]:
incoming_df = fetch_missing_intervals(
    missing_intervals
)

if incoming_df is not None:
    incoming_shape = incoming_df.shape
else:
    incoming_shape = None

inserted_rows = 0

if incoming_df is not None:
    with psycopg.connect(
        dbname=DATABASE_NAME
    ) as conn:
        inserted_rows = load_canonical(
            conn,
            incoming_df,
        )

        validate_coverage(
            conn,
            PRACTICE_START_LOCAL,
            PRACTICE_END_LOCAL,
        )

incoming_shape, inserted_rows

Fetching: 2024-01-01 00:00:00-07:00 to 2025-01-01 00:00:00-07:00


((8784, 16), 8784)

In [16]:
# practice table containing only the required interval

def materialize_practice_table(conn):
    (
        start_utc,
        end_utc,
        _,
    ) = interval_contract(
        PRACTICE_START_LOCAL,
        PRACTICE_END_LOCAL,
    )

    conn.execute(
        sql.SQL('''
            CREATE SCHEMA IF NOT EXISTS {}
        ''').format(
            sql.Identifier(
                PRACTICE_SCHEMA
            )
        )
    )

    conn.execute(
        sql.SQL('''
            DROP TABLE IF EXISTS {}.{}
        ''').format(
            sql.Identifier(
                PRACTICE_SCHEMA
            ),
            sql.Identifier(
                PRACTICE_TABLE
            ),
        )
    )

    column_sql = sql.SQL(", ").join(
        sql.Identifier(column)
        for column in COLUMNS
    )

    conn.execute(
        sql.SQL('''
            CREATE TABLE {}.{} AS
            SELECT {}
            FROM {}.{}
            WHERE time >= %s
              AND time < %s
        ''').format(
            sql.Identifier(
                PRACTICE_SCHEMA
            ),
            sql.Identifier(
                PRACTICE_TABLE
            ),
            column_sql,
            sql.Identifier(
                CANONICAL_SCHEMA
            ),
            sql.Identifier(
                CANONICAL_TABLE
            ),
        ),
        (
            start_utc
            .to_pydatetime(),
            end_utc
            .to_pydatetime(),
        ),
    )


def validate_practice_table(conn):
    (
        _,
        _,
        expected_times,
    ) = interval_contract(
        PRACTICE_START_LOCAL,
        PRACTICE_END_LOCAL,
    )

    summary = (
        conn.execute(
            sql.SQL('''
                SELECT
                    COUNT(*),
                    MIN(time),
                    MAX(time)
                FROM {}.{}
            ''').format(
                sql.Identifier(
                    PRACTICE_SCHEMA
                ),
                sql.Identifier(
                    PRACTICE_TABLE
                ),
            )
        )
        .fetchone()
    )

    if (
        summary[0]
        != len(expected_times)
    ):
        raise ValueError(
            "Practice row-count validation failed."
        )

    actual_start = (
        pd.Timestamp(
            summary[1]
        )
        .tz_convert("UTC")
    )

    actual_end = (
        pd.Timestamp(
            summary[2]
        )
        .tz_convert("UTC")
    )

    if (
        actual_start
        != expected_times[0]
        or actual_end
        != expected_times[-1]
    ):
        raise ValueError(
            "Practice boundary validation failed."
        )

    return summary

In [17]:
with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    materialize_practice_table(
        conn
    )

    practice_summary = (
        validate_practice_table(
            conn
        )
    )

with psycopg.connect(
    dbname=DATABASE_NAME
) as conn:
    rows = conn.execute(
        sql.SQL('''
            SELECT
                time,
                precipitation
            FROM {}.{}
            ORDER BY time
            LIMIT 5
        ''').format(
            sql.Identifier(
                PRACTICE_SCHEMA
            ),
            sql.Identifier(
                PRACTICE_TABLE
            ),
        )
    ).fetchall()

practice_summary, pd.DataFrame(
    rows,
    columns=[
        "time",
        "precipitation",
    ],
)

((17544,
  datetime.datetime(2024, 1, 1, 7, 0, tzinfo=datetime.timezone.utc),
  datetime.datetime(2026, 1, 1, 6, 0, tzinfo=datetime.timezone.utc)),
                        time  precipitation
 0 2024-01-01 07:00:00+00:00            0.6
 1 2024-01-01 08:00:00+00:00            0.8
 2 2024-01-01 09:00:00+00:00            0.8
 3 2024-01-01 10:00:00+00:00            0.5
 4 2024-01-01 11:00:00+00:00            0.2)

In [18]:
def ensure_database():
    with psycopg.connect(
        dbname="postgres",
        autocommit=True,
    ) as conn:
        exists = conn.execute(
            '''
            SELECT EXISTS (
                SELECT 1
                FROM pg_database
                WHERE datname = %s
            );
            ''',
            (DATABASE_NAME,),
        ).fetchone()[0]

        if not exists:
            conn.execute(
                sql.SQL('''
                    CREATE DATABASE {}
                ''').format(
                    sql.Identifier(
                        DATABASE_NAME
                    )
                )
            )

        conn.execute(
            sql.SQL('''
                ALTER DATABASE {}
                SET timezone TO {}
            ''').format(
                sql.Identifier(
                    DATABASE_NAME
                ),
                sql.Literal("UTC"),
            )
        )


def main():
    ensure_database()

    with psycopg.connect(
        dbname=DATABASE_NAME
    ) as conn:
        ensure_canonical_storage(
            conn
        )

        validate_canonical_storage(
            conn
        )

        missing_intervals = (
            find_missing_intervals(
                conn,
                PRACTICE_START_LOCAL,
                PRACTICE_END_LOCAL,
            )
        )

    incoming_df = (
        fetch_missing_intervals(
            missing_intervals
        )
    )

    with psycopg.connect(
        dbname=DATABASE_NAME
    ) as conn:
        inserted_rows = 0

        if incoming_df is not None:
            inserted_rows = (
                load_canonical(
                    conn,
                    incoming_df,
                )
            )

        validate_coverage(
            conn,
            PRACTICE_START_LOCAL,
            PRACTICE_END_LOCAL,
        )

        materialize_practice_table(
            conn
        )

        practice_summary = (
            validate_practice_table(
                conn
            )
        )

    print(
        "Canonical rows inserted:",
        inserted_rows,
    )

    print(
        "Practice table:",
        f"{PRACTICE_SCHEMA}."
        f"{PRACTICE_TABLE}",
    )

    print(
        "Practice summary:",
        practice_summary,
    )

In [19]:
# the first run should create the required canonical coverage and practice table. The second should fetch nothing and insert no canonical rows.

reset_dev_database()

print("FIRST RUN")
main()

print("\nSECOND RUN")
main()

FIRST RUN
Fetching: 2024-01-01 00:00:00-07:00 to 2025-01-01 00:00:00-07:00
Fetching: 2025-01-01 00:00:00-07:00 to 2026-01-01 00:00:00-07:00
Canonical rows inserted: 17544
Practice table: practice_008.weather_hourly
Practice summary: (17544, datetime.datetime(2024, 1, 1, 7, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 1, 1, 6, 0, tzinfo=datetime.timezone.utc))

SECOND RUN
Canonical rows inserted: 0
Practice table: practice_008.weather_hourly
Practice summary: (17544, datetime.datetime(2024, 1, 1, 7, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 1, 1, 6, 0, tzinfo=datetime.timezone.utc))
